In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from PIL import Image

In [7]:
class NormalizedBBoxRegressor(nn.Module):
    """
    归一化边界框回归模型
    输入: (batch_size, 64, 64, 256) 的特征图
    输出: 归一化边界框 (x, y, w, h) 范围[0, 1]
    """
    
    def __init__(self, input_channels=256, img_width=1024, img_height=1024):
        super(NormalizedBBoxRegressor, self).__init__()
        
        self.input_channels = input_channels
        self.img_width = img_width
        self.img_height = img_height
        
        # 特征处理器 - 一层卷积
        self.feature_processor = nn.Sequential(
            # 输入: (batch_size, 256, 64, 64)
            nn.Conv2d(input_channels, 512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 64x64 -> 32x32
            
            # 可选的第二层
            nn.Conv2d(512, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32 -> 16x16
        )
        
        # 全局平均池化
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 回归头 - 输出归一化坐标
        self.regressor = nn.Sequential(
            nn.Linear(256, 128),  # 输入维度从512改为256
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 4)  # 输出: x, y, w, h (归一化坐标 0-1)
        )
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征图, shape: (batch_size, 64, 64, 256) 或 (batch_size, 256, 64, 64)
        
        返回:
            bbox: 归一化边界框, shape: (batch_size, 4)
                  格式: [x_center, y_center, width, height] (归一化坐标 0-1)
        """
        # 确保输入形状正确 (NCHW格式)
        if x.dim() == 4:
            if x.shape[-1] == 256:  # NHWC格式
                x = x.permute(0, 3, 1, 2)  # 转换为NCHW: (batch, 256, 64, 64)
        else:
            raise ValueError(f"输入维度错误: {x.shape}")
        
        # 特征处理
        features = self.feature_processor(x)  # (batch, 256, 16, 16)
        
        # 全局池化
        features = self.global_pool(features)  # (batch, 256, 1, 1)
        
        # 展平
        features = features.view(features.size(0), -1)  # (batch, 256)
        
        # 回归
        bbox = self.regressor(features)  # (batch, 4)
        
        # 使用sigmoid确保输出在0-1之间
        bbox = torch.sigmoid(bbox)
        
        return bbox
    
    def normalize_coords(self, pixel_coords):
        """
        将像素坐标归一化
        
        参数:
            pixel_coords: 像素坐标 [x, y, w, h]
        
        返回:
            normalized_coords: 归一化坐标 [x, y, w, h] 范围[0, 1]
        """
        normalized = pixel_coords.clone()
        if len(normalized.shape) == 1:
            normalized = normalized.unsqueeze(0)
        
        # 归一化
        normalized[:, 0] = pixel_coords[:, 0] / self.img_width   # x
        normalized[:, 1] = pixel_coords[:, 1] / self.img_height  # y
        normalized[:, 2] = pixel_coords[:, 2] / self.img_width   # w
        normalized[:, 3] = pixel_coords[:, 3] / self.img_height  # h
        
        return normalized.squeeze()
    
    def denormalize_coords(self, normalized_coords):
        """
        将归一化坐标转换为像素坐标
        
        参数:
            normalized_coords: 归一化坐标 [x, y, w, h] 范围[0, 1]
        
        返回:
            pixel_coords: 像素坐标
        """
        pixel = normalized_coords.clone()
        if len(pixel.shape) == 1:
            pixel = pixel.unsqueeze(0)
        
        # 反归一化
        pixel[:, 0] = normalized_coords[:, 0] * self.img_width   # x
        pixel[:, 1] = normalized_coords[:, 1] * self.img_height  # y
        pixel[:, 2] = normalized_coords[:, 2] * self.img_width   # w
        pixel[:, 3] = normalized_coords[:, 3] * self.img_height  # h
        
        return pixel.squeeze()

class NormalizedBBoxLoss(nn.Module):
    """
    归一化坐标的边界框损失函数
    """
    
    def __init__(self, lambda_l1=1.0, lambda_iou=3.0):
        super(NormalizedBBoxLoss, self).__init__()
        self.lambda_l1 = lambda_l1
        self.lambda_iou = lambda_iou
        
    def forward(self, pred, target):
        """
        计算损失
        
        参数:
            pred: 预测归一化坐标 [x, y, w, h] (0-1)
            target: 真实归一化坐标 [x, y, w, h] (0-1)
        
        返回:
            loss_dict: 各种损失组成的字典
        """
        # 1. L1损失
        l1_loss = F.l1_loss(pred, target)
        
        # 2. IoU损失
        iou_loss = self.iou_loss(pred, target)
        
        # 4. 中心点约束损失
        center_loss = self.center_constraint_loss(pred)
        
        # 总损失
        total_loss = (
            self.lambda_l1 * l1_loss +
            self.lambda_iou * iou_loss +
            0.1 * center_loss
        )
        
        return {
            'total_loss': total_loss,
            'l1_loss': l1_loss,
            'iou_loss': iou_loss,
            'center_loss': center_loss
        }
    
    def iou_loss(self, pred, target):
        """归一化坐标的IoU损失"""
        # 转换为角点坐标
        pred_corners = self.to_corners(pred)
        target_corners = self.to_corners(target)
        
        # 计算交集
        inter_x1 = torch.max(pred_corners[:, 0], target_corners[:, 0])
        inter_y1 = torch.max(pred_corners[:, 1], target_corners[:, 1])
        inter_x2 = torch.min(pred_corners[:, 2], target_corners[:, 2])
        inter_y2 = torch.min(pred_corners[:, 3], target_corners[:, 3])
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        
        # 计算并集
        pred_area = (pred_corners[:, 2] - pred_corners[:, 0]) * (pred_corners[:, 3] - pred_corners[:, 1])
        target_area = (target_corners[:, 2] - target_corners[:, 0]) * (target_corners[:, 3] - target_corners[:, 1])
        union_area = pred_area + target_area - inter_area + 1e-6
        
        iou = inter_area / union_area
        iou_loss = 1.0 - iou
        
        return iou_loss.mean()
    
    def to_corners(self, boxes):
        """归一化坐标转角点坐标"""
        x, y, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2
        
        return torch.stack([x1, y1, x2, y2], dim=1)
    
    def center_constraint_loss(self, pred):
        """中心点约束损失"""
        x, y = pred[:, 0], pred[:, 1]
        
        # 鼓励中心点在图像内
        center_loss = torch.relu(-x) + torch.relu(x - 1) + torch.relu(-y) + torch.relu(y - 1)
        
        return center_loss.mean()

class NormalizedDataset(Dataset):
    """
    归一化坐标数据集
    """
    
    def __init__(self, feature_files, box_files, feature_folder, img_width=1024, img_height=1024):
        """
        初始化
        
        参数:
            feature_files: 特征文件列表
            box_files: 边界框文件列表
            feature_folder: 特征文件夹路径
            img_width: 图像宽度
            img_height: 图像高度
        """
        self.feature_files = feature_files
        self.box_files = box_files
        self.feature_folder = feature_folder
        self.img_width = img_width
        self.img_height = img_height
        
        # 预加载边界框
        self.boxes = []
        for box_file in box_files:
            with open(box_file, "r") as f:
                cor = list(map(int, f.readline().strip().split(" ")))
                if len(cor) == 4:  # 确保是x,y,w,h格式
                    self.boxes.append(cor)
                elif len(cor) == 8:  # 如果是x1,y1,x2,y2,x3,y3,x4,y4格式
                    # 转换为x,y,w,h
                    xs = [cor[0], cor[2], cor[4], cor[6]]
                    ys = [cor[1], cor[3], cor[5], cor[7]]
                    x_center = sum(xs) / 4
                    y_center = sum(ys) / 4
                    width = max(xs) - min(xs)
                    height = max(ys) - min(ys)
                    self.boxes.append([x_center, y_center, width, height])
        
        # 确保文件数量匹配
        assert len(self.feature_files) == len(self.boxes), \
            f"特征文件数 ({len(self.feature_files)}) 不等于边界框数 ({len(self.boxes)})"
    
    def __len__(self):
        return len(self.feature_files)
    
    def __getitem__(self, idx):
        # 加载特征
        feature_path = os.path.join(self.feature_folder, self.feature_files[idx])
        features = torch.load(feature_path)
        
        # 确保特征形状
        if features.dim() == 3:  # (64, 64, 256)
            pass  # 保持原状
        elif features.dim() == 4:  # (1, 256, 64, 64)或其他
            if features.shape[0] == 1:
                features = features.squeeze(0)
            if features.shape[0] == 256:  # (256, 64, 64)
                features = features.permute(1, 2, 0)  # 转换为(64, 64, 256)
        
        # 获取边界框并归一化
        pixel_bbox = torch.tensor(self.boxes[idx], dtype=torch.float32)
        
        # 归一化
        normalized_bbox = torch.zeros_like(pixel_bbox)
        normalized_bbox[0] = pixel_bbox[0] / self.img_width   # x
        normalized_bbox[1] = pixel_bbox[1] / self.img_height  # y
        normalized_bbox[2] = pixel_bbox[2] / self.img_width   # w
        normalized_bbox[3] = pixel_bbox[3] / self.img_height  # h
        
        return features.float(), normalized_bbox

In [8]:
class BBoxTrainer:
    """
    边界框训练器
    """
    
    def __init__(self, model, device='cuda:0', lr=1e-3, weight_decay=1e-4):

        device_ids = list(range(torch.cuda.device_count()))
        device = torch.device(f"cuda:{device_ids[0]}" if device_ids else "cpu")
        
        print(f"可用 GPU: {device_ids}")
        print(f"main GPU is device {device_ids[0]}")
        
        self.model = nn.DataParallel(model.to(device), device_ids=device_ids)
        self.device = device
        self.lr = lr
        self.weight_decay = weight_decay
        
        # 损失函数
        self.criterion = NormalizedBBoxLoss(
            lambda_l1=1.0,
            lambda_iou=3.0
        )
        
        # 优化器
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        
        # 学习率调度器
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )
        
        # 训练历史
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_iou': [], 'val_iou': [],
            'train_l1': [], 'val_l1': []
        }
    
    def train_epoch(self, train_loader, epoch, verbose=True):
        """训练一个epoch"""
        self.model.train()
        total_loss = 0
        total_iou = 0
        total_l1 = 0
        num_batches = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', disable=not verbose)
        
        for features, targets in pbar:
            # 移动到设备
            features = features.to(self.device)  # (B, 64, 64, 256)
            targets = targets.to(self.device)    # (B, 4) 归一化坐标
            
            # 前向传播
            predictions = self.model(features)   # (B, 4) 归一化坐标
            
            # 计算损失
            loss_dict = self.criterion(predictions, targets)
            loss = loss_dict['total_loss']
            
            # 反向传播
            self.optimizer.zero_grad()
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            # 更新参数
            self.optimizer.step()
            
            # 计算IoU
            batch_iou = self.calculate_iou(predictions, targets)
            
            # 计算L1误差
            batch_l1 = F.l1_loss(predictions, targets).item()
            
            # 累加统计
            total_loss += loss.item()
            total_iou += batch_iou.mean().item()
            total_l1 += batch_l1
            num_batches += 1
            
            # 更新进度条
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'iou': f'{batch_iou.mean().item():.4f}',
                'l1': f'{batch_l1:.4f}',
                'lr': f'{self.optimizer.param_groups[0]["lr"]:.6f}'
            })
        
        avg_loss = total_loss / num_batches
        avg_iou = total_iou / num_batches
        avg_l1 = total_l1 / num_batches
        
        return avg_loss, avg_iou, avg_l1
    
    def validate(self, val_loader, verbose=True):
        """验证"""
        self.model.eval()
        total_loss = 0
        total_iou = 0
        total_l1 = 0
        num_batches = 0
        
        all_predictions = []
        all_targets = []
        
        with torch.no_grad():
            pbar = tqdm(val_loader, desc='[Validation]', disable=not verbose)
            
            for features, targets in pbar:
                # 移动到设备
                features = features.to(self.device)
                targets = targets.to(self.device)
                
                # 前向传播
                predictions = self.model(features)
                
                # 计算损失
                loss_dict = self.criterion(predictions, targets)
                loss = loss_dict['total_loss']
                
                # 计算IoU
                batch_iou = self.calculate_iou(predictions, targets)
                
                # 计算L1误差
                batch_l1 = F.l1_loss(predictions, targets).item()
                
                # 累加统计
                total_loss += loss.item()
                total_iou += batch_iou.mean().item()
                total_l1 += batch_l1
                num_batches += 1
                
                # 保存预测和真实值
                all_predictions.append(predictions.cpu())
                all_targets.append(targets.cpu())
                
                # 更新进度条
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'iou': f'{batch_iou.mean().item():.4f}',
                    'l1': f'{batch_l1:.4f}'
                })
        
        avg_loss = total_loss / num_batches
        avg_iou = total_iou / num_batches
        avg_l1 = total_l1 / num_batches
        
        # 合并所有预测
        all_predictions = torch.cat(all_predictions, dim=0)
        all_targets = torch.cat(all_targets, dim=0)
        
        return avg_loss, avg_iou, avg_l1, all_predictions, all_targets
    
    def calculate_iou(self, pred, target):
        """计算IoU"""
        # 转换为角点坐标
        def to_corners(boxes):
            x, y, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
            x1 = x - w / 2
            y1 = y - h / 2
            x2 = x + w / 2
            y2 = y + h / 2
            return torch.stack([x1, y1, x2, y2], dim=1)
        
        pred_corners = to_corners(pred)
        target_corners = to_corners(target)
        
        # 计算交集
        inter_x1 = torch.max(pred_corners[:, 0], target_corners[:, 0])
        inter_y1 = torch.max(pred_corners[:, 1], target_corners[:, 1])
        inter_x2 = torch.min(pred_corners[:, 2], target_corners[:, 2])
        inter_y2 = torch.min(pred_corners[:, 3], target_corners[:, 3])
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        
        # 计算并集
        pred_area = (pred_corners[:, 2] - pred_corners[:, 0]) * (pred_corners[:, 3] - pred_corners[:, 1])
        target_area = (target_corners[:, 2] - target_corners[:, 0]) * (target_corners[:, 3] - target_corners[:, 1])
        union_area = pred_area + target_area - inter_area + 1e-6
        
        iou = inter_area / union_area
        return iou
    
    def train(self, train_loader, val_loader, epochs=70, 
              save_path='best_bbox_model.pth', verbose=True):
        """完整训练过程"""
        best_val_iou = 0
        
        for epoch in range(1, epochs + 1):
            if verbose:
                print(f"\n{'='*60}")
                print(f"Epoch {epoch}/{epochs}")
                print('='*60)
            
            # 训练
            train_loss, train_iou, train_l1 = self.train_epoch(train_loader, epoch, verbose)
            
            # 验证
            val_loss, val_iou, val_l1, val_preds, val_targets = self.validate(val_loader, verbose)
            
            # 更新学习率
            self.scheduler.step(val_loss)
            
            # 保存历史
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_iou'].append(train_iou)
            self.history['val_iou'].append(val_iou)
            self.history['train_l1'].append(train_l1)
            self.history['val_l1'].append(val_l1)
            
            # 保存最佳模型
            if val_iou > best_val_iou:
                best_val_iou = val_iou
                self.save_model(save_path)
                if verbose:
                    print(f"✨ 保存最佳模型，验证IoU: {val_iou:.4f}")
            
            # 打印epoch总结
            if verbose:
                print(f"\nEpoch {epoch} 总结:")
                print(f"  训练损失: {train_loss:.4f}, 训练IoU: {train_iou:.4f}, 训练L1: {train_l1:.4f}")
                print(f"  验证损失: {val_loss:.4f}, 验证IoU: {val_iou:.4f}, 验证L1: {val_l1:.4f}")
                print(f"  最佳验证IoU: {best_val_iou:.4f}")
                print(f"  学习率: {self.optimizer.param_groups[0]['lr']:.6f}")
        
        # 加载最佳模型
        self.load_model(save_path)
        
        return self.history
    
    def save_model(self, path):
        """保存模型"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'history': self.history,
            'img_width': self.model.module.img_width,
            'img_height': self.model.module.img_height
        }, path)
    
    def load_model(self, path):
        """加载模型"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        if 'optimizer_state_dict' in checkpoint:
            self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if 'history' in checkpoint:
            self.history = checkpoint['history']
    
    def plot_training_history(self):
        """绘制训练历史"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 损失曲线
        axes[0, 0].plot(self.history['train_loss'], label='Train loss', linewidth=2)
        axes[0, 0].plot(self.history['val_loss'], label='Val loss', linewidth=2)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Train and Val Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # IoU曲线
        axes[0, 1].plot(self.history['train_iou'], label='Train IoU', linewidth=2)
        axes[0, 1].plot(self.history['val_iou'], label='Val IoU', linewidth=2)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('IoU')
        axes[0, 1].set_title('train and val IoU')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # L1误差曲线
        axes[1, 0].plot(self.history['train_l1'], label="train L1 loss", linewidth=2)
        axes[1, 0].plot(self.history['val_l1'], label='val L1 loss', linewidth=2)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('L1 loss (Normalized)')
        axes[1, 0].set_title('Train and Val L1 Loss')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 最佳验证IoU
        best_val_iou = max(self.history['val_iou']) if self.history['val_iou'] else 0
        axes[1, 1].bar(['Best Val IoU'], [best_val_iou], color='skyblue')
        axes[1, 1].set_ylabel('IoU')
        axes[1, 1].set_title(f'Best Val IoU: {best_val_iou:.4f}')
        axes[1, 1].set_ylim(0, 1)
        axes[1, 1].grid(True, alpha=0.3, axis='y')
        
        plt.suptitle('Train History', fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

In [9]:
def file_generator(folder_path):
    """
    文件路径生成器
    """
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            yield os.path.join(root, file)

def prepare_data():
    """准备数据"""
    # 你的数据路径
    datafolder = "../SAM_mobile_features/image_feature"
    box_folder = "../datasets"
    
    # 获取特征文件
    files = os.listdir(datafolder)
    file_sort = sorted(files)
    
    # 获取边界框文件
    boxes = [x for x in file_generator(box_folder) if ".txt" in x]
    box_sort = sorted(boxes)
    
    # 确保数量匹配
    min_len = min(len(file_sort), len(box_sort))
    file_sort = file_sort[:min_len]
    box_sort = box_sort[:min_len]
    
    print(f"特征文件数: {len(file_sort)}")
    print(f"边界框文件数: {len(box_sort)}")
    
    # 分割数据集
    X_train, X_val, y_train, y_val = train_test_split(
        file_sort, box_sort,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )
    
    # 创建数据集
    train_data = NormalizedDataset(
        feature_files=X_train,
        box_files=y_train,
        feature_folder=datafolder,
        img_width=1024,
        img_height=1024
    )
    
    val_data = NormalizedDataset(
        feature_files=X_val,
        box_files=y_val,
        feature_folder=datafolder,
        img_width=1024,
        img_height=1024
    )
    
    print(f"训练集大小: {len(train_data)}")
    print(f"验证集大小: {len(val_data)}")
    
    return train_data, val_data, X_val, y_val

def train_model():
    """训练模型"""
    # 设置随机种子
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 检查设备
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    # 准备数据
    print("准备数据...")
    train_data, val_data, val_files, val_boxes = prepare_data()
    
    # 创建数据加载器
    train_loader = DataLoader(
        train_data, 
        batch_size=128, 
        shuffle=True, 
        num_workers=4,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_data, 
        batch_size=128, 
        shuffle=False, 
        num_workers=4,
        pin_memory=True
    )
    
    # 创建模型
    print("\n创建模型...")
    model = NormalizedBBoxRegressor(
        input_channels=256,
        img_width=1024,
        img_height=1024
    )
    
    # 打印模型信息
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
    print(f"可训练参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # 测试前向传播
    with torch.no_grad():
        # 获取一个样本测试
        sample_features, sample_bbox = train_data[0]
        sample_features = sample_features.unsqueeze(0)  # 增加batch维度
        sample_output = model(sample_features)
        
        print(f"\n样本测试:")
        print(f"输入形状: {sample_features.shape}")
        print(f"输出形状: {sample_output.shape}")
        print(f"预测归一化坐标: {sample_output[0].numpy()}")
        print(f"真实归一化坐标: {sample_bbox.numpy()}")
    
    # 创建训练器
    trainer = BBoxTrainer(
        model,
        device=device,
        lr=1e-3
    )
    
    # 训练模型
    print("\n开始训练...")
    history = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=70,
        save_path='best_mobile_normalized_bbox_model.pth',
        verbose=True
    )
    
    # 绘制训练历史
    print("\n绘制训练历史...")
    trainer.plot_training_history()
    
    return model, trainer, val_loader, val_files, val_boxes

def test_model(model, val_loader, val_files, device='cuda'):
    """测试模型"""
    model.eval()
    
    print("\n" + "="*60)
    print("模型测试结果")
    print("="*60)
    
    # 验证
    val_loss, val_iou, val_l1, predictions, targets = validate_model(
        model, val_loader, device
    )
    
    print(f"验证集IoU: {val_iou:.4f}")
    print(f"验证集L1误差: {val_l1:.4f}")
    print(f"验证集损失: {val_loss:.4f}")
    
    # 转换为像素坐标
    pixel_predictions = model.denormalize_coords(predictions)
    pixel_targets = model.denormalize_coords(targets)
    
    # 计算像素级别的误差
    pixel_l1 = F.l1_loss(pixel_predictions, pixel_targets).item()
    pixel_iou = calculate_pixel_iou(pixel_predictions, pixel_targets)
    
    print(f"\n像素级别指标:")
    print(f"像素L1误差: {pixel_l1:.1f} 像素")
    print(f"像素IoU: {pixel_iou.mean().item():.4f}")
    
    # 显示前5个样本的结果
    print(f"\n前5个样本的详细结果:")
    keys = []
    select_index = []
    selected_files = []
    selected_predict = []
    selected_targetes = []
    for i in range(len(val_files)):
        element = val_files[i][:-3].split("_set_file_")[0]
        if element in keys:
            pass
        else:
            keys.append(element)
            select_index.append(i)
            selected_files.append(val_files[i])
            selected_predict.append(pixel_predictions[i].numpy())
            selected_targetes.append(pixel_targets[i].numpy())
        
        if len(keys) == 5:
            break

    print(keys)      
    print(select_index)
    print(selected_files)
    
    for i in range(min(5, len(val_files))):
        print(f"\nSample {i+1}: {val_files[i]}")
        print(f"  预测像素坐标: {pixel_predictions[i].numpy()}")
        print(f"  真实像素坐标: {pixel_targets[i].numpy()}")
        print(f"  归一化坐标: {predictions[i].numpy()}")
        print(f"  IoU: {pixel_iou[i].item():.4f}")
        print(f"  L1误差: {F.l1_loss(pixel_predictions[i], pixel_targets[i]).item():.1f} 像素")
    
    # 可视化预测
    visualize_predictions(selected_files, pixel_predictions[select_index], pixel_targets[select_index])
    
    return {
        'normalized_iou': val_iou,
        'normalized_l1': val_l1,
        'pixel_iou': pixel_iou.mean().item(),
        'pixel_l1': pixel_l1,
        'predictions': predictions,
        'targets': targets
    }

def validate_model(model, val_loader, device):
    
    """验证模型"""
    model.eval()
    total_loss = 0
    total_iou = 0
    total_l1 = 0
    num_batches = 0
    
    all_predictions = []
    all_targets = []
    
    criterion = NormalizedBBoxLoss()
    
    with torch.no_grad():
        for features, targets in tqdm(val_loader, desc='测试'):
            features = features.to(device)
            targets = targets.to(device)
            
            predictions = model(features)
            
            # 计算损失
            loss_dict = criterion(predictions, targets)
            loss = loss_dict['total_loss']
            
            # 计算IoU
            batch_iou = calculate_iou(predictions, targets)
            
            # 计算L1误差
            batch_l1 = F.l1_loss(predictions, targets).item()
            
            # 累加统计
            total_loss += loss.item()
            total_iou += batch_iou.mean().item()
            total_l1 += batch_l1
            num_batches += 1
            
            # 保存结果
            all_predictions.append(predictions.cpu())
            all_targets.append(targets.cpu())
    
    avg_loss = total_loss / num_batches
    avg_iou = total_iou / num_batches
    avg_l1 = total_l1 / num_batches
    
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    return avg_loss, avg_iou, avg_l1, all_predictions, all_targets

def calculate_iou(pred, target):
    """计算IoU"""
    def to_corners(boxes):
        x, y, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2
        return torch.stack([x1, y1, x2, y2], dim=1)
    
    pred_corners = to_corners(pred)
    target_corners = to_corners(target)
    
    inter_x1 = torch.max(pred_corners[:, 0], target_corners[:, 0])
    inter_y1 = torch.max(pred_corners[:, 1], target_corners[:, 1])
    inter_x2 = torch.min(pred_corners[:, 2], target_corners[:, 2])
    inter_y2 = torch.min(pred_corners[:, 3], target_corners[:, 3])
    
    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
    
    pred_area = (pred_corners[:, 2] - pred_corners[:, 0]) * (pred_corners[:, 3] - pred_corners[:, 1])
    target_area = (target_corners[:, 2] - target_corners[:, 0]) * (target_corners[:, 3] - target_corners[:, 1])
    union_area = pred_area + target_area - inter_area + 1e-6
    
    iou = inter_area / union_area
    return iou

def calculate_pixel_iou(pred, target):
    """计算像素级别的IoU"""
    return calculate_iou(pred, target)

def visualize_predictions(val_files, pixel_predictions, pixel_targets, img_width=1024, img_height=1024):
    """可视化预测结果"""
    pixel_predictions = pixel_predictions.numpy()
    pixel_targets = pixel_targets.numpy()
    
    num_samples = min(5, len(pixel_predictions))
    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 5, 5))
    
    for i in range(num_samples):
        if num_samples > 1:
            ax = axes[i]
        else:
            ax = axes
        
        # 预测框
        pred_box = pixel_predictions[i]
        # 真实框
        target_box = pixel_targets[i]

        file_path = val_files[i]
        # print(file_path)
        # print(file_path[:-3].split("_set_file_"))
        element = file_path[:-3].split("_set_file_")
        checks = ["jpg", "png"]
        img_path = ""
        for extend in checks:
            file = f"../datasets/{element[0]}/image/{element[1]}.{extend}"
            if os.path.isfile(file):
                img_path = file
                break

        ## Darwin_set_file_00005054_NORMAL2-IM-0543-0001.pt

        image = Image.open(img_path).convert('RGB')
        image = image.resize((1024, 1024), Image.BICUBIC)
        image = np.array(image)
        
        # 计算IoU
        def box_iou(box1, box2):
            x1 = max(box1[0] - box1[2]/2, box2[0] - box2[2]/2)
            y1 = max(box1[1] - box1[3]/2, box2[1] - box2[3]/2)
            x2 = min(box1[0] + box1[2]/2, box2[0] + box2[2]/2)
            y2 = min(box1[1] + box1[3]/2, box2[1] + box2[3]/2)
            
            inter = max(0, x2 - x1) * max(0, y2 - y1)
            area1 = box1[2] * box1[3]
            area2 = box2[2] * box2[3]
            union = area1 + area2 - inter
            
            return inter / (union + 1e-6) if union > 0 else 0
        
        iou = box_iou(pred_box, target_box)

        # 创建空白图像
        ax.imshow(image)
        
        # 绘制真实框 (绿色)
        x1 = target_box[0] - target_box[2]/2
        y1 = target_box[1] - target_box[3]/2
        width = target_box[2]
        height = target_box[3]
        rect = plt.Rectangle((x1, y1), width, height, 
                           linewidth=2, edgecolor='green', facecolor='none')
        ax.add_patch(rect)
        
        # 绘制预测框 (红色)
        x1_pred = pred_box[0] - pred_box[2]/2
        y1_pred = pred_box[1] - pred_box[3]/2
        width_pred = pred_box[2]
        height_pred = pred_box[3]
        rect_pred = plt.Rectangle((x1_pred, y1_pred), width_pred, height_pred,
                                 linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
        ax.add_patch(rect_pred)
        
        ax.set_xlim(0, img_width)
        ax.set_ylim(img_height, 0)  # 反转y轴
        ax.set_title(f'Sample {i+1}\nIoU: {iou:.3f}')
        ax.axis('off')
    
    plt.suptitle('Predicted Result (Pixel Level)', fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

def inference_example(model, device='cuda'):
    """推理示例"""
    model.eval()
    
    # 加载测试数据
    datafolder = "../SAM_mobile_features/image_feature"
    test_files = os.listdir(datafolder)[:10]  # 取前3个文件测试
    
    print("\n推理示例:")
    print("="*60)
    
    for i, file in enumerate(test_files):
        # 加载特征
        feature_path = os.path.join(datafolder, file)
        features = torch.load(feature_path)
        
        # 确保形状正确
        if features.dim() == 3:  # (64, 64, 256)
            features = features.unsqueeze(0)  # 增加batch维度
        elif features.dim() == 4 and features.shape[0] == 1:
            pass  # 已经是(batch, ...)格式
        
        features = features.to(device)
        
        # 推理
        with torch.no_grad():
            normalized_pred = model(features)
            pixel_pred = model.denormalize_coords(normalized_pred)
        
        print(f"\nSample {i+1}: {file}")
        print(f"  Input shape: {features.shape}")
        print(f"  Predicted Normalized box: {normalized_pred[0].cpu().numpy()}")
        print(f"  Predicted Pixel box: {pixel_pred.cpu().numpy()}")



In [10]:
# # 测试
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# test_results = test_model(model, val_loader, val_files, device)

In [ ]:
# 训练
model, trainer, val_loader, val_files, val_boxes = train_model()

使用设备: cuda:0
准备数据...
特征文件数: 11882
边界框文件数: 11882
训练集大小: 9505
验证集大小: 2377

创建模型...
模型参数量: 2,403,012
可训练参数量: 2,403,012

样本测试:
输入形状: torch.Size([1, 256, 64, 64])
输出形状: torch.Size([1, 4])
预测归一化坐标: [0.4997842  0.5000162  0.50033885 0.49985072]
真实归一化坐标: [0.5        0.5332031  0.7939453  0.92285156]
可用 GPU: [0, 1]
main GPU is device 0

开始训练...

Epoch 1/70


Epoch 1 [Train]: 100%|█| 75/75 [00:21<00:00,  3.54it/s, loss=1.0445, iou=0.6766,
[Validation]: 100%|█| 19/19 [00:05<00:00,  3.22it/s, loss=1.1811, iou=0.6355, l1


✨ 保存最佳模型，验证IoU: 0.6266

Epoch 1 总结:
  训练损失: 1.2372, 训练IoU: 0.6198, 训练L1: 0.0967
  验证损失: 1.2083, 验证IoU: 0.6266, 验证L1: 0.0881
  最佳验证IoU: 0.6266
  学习率: 0.001000

Epoch 2/70


Epoch 2 [Train]: 100%|█| 75/75 [00:20<00:00,  3.59it/s, loss=1.1156, iou=0.6549,
[Validation]: 100%|█| 19/19 [00:06<00:00,  3.04it/s, loss=1.0293, iou=0.6820, l1


✨ 保存最佳模型，验证IoU: 0.6668

Epoch 2 总结:
  训练损失: 1.0693, 训练IoU: 0.6705, 训练L1: 0.0808
  验证损失: 1.0765, 验证IoU: 0.6668, 验证L1: 0.0768
  最佳验证IoU: 0.6668
  学习率: 0.001000

Epoch 3/70


...

Epoch 47 [Train]: 100%|█| 75/75 [00:24<00:00,  3.10it/s, loss=0.7526, iou=0.7630
[Validation]: 100%|█| 19/19 [00:05<00:00,  3.17it/s, loss=0.7485, iou=0.7659, l1



Epoch 47 总结:
  训练损失: 0.7928, 训练IoU: 0.7519, 训练L1: 0.0485
  验证损失: 0.7973, 验证IoU: 0.7509, 验证L1: 0.0502
  最佳验证IoU: 0.7518
  学习率: 0.000250

Epoch 48/70


Epoch 48 [Train]: 100%|█| 75/75 [00:20<00:00,  3.61it/s, loss=0.8330, iou=0.7397
[Validation]: 100%|█| 19/19 [00:06<00:00,  2.81it/s, loss=0.7687, iou=0.7591, l1



Epoch 48 总结:
  训练损失: 0.7862, 训练IoU: 0.7539, 训练L1: 0.0479
  验证损失: 0.8028, 验证IoU: 0.7490, 验证L1: 0.0498
  最佳验证IoU: 0.7518
  学习率: 0.000250

Epoch 49/70


Epoch 49 [Train]:  35%|▎| 26/75 [00:10<00:11,  4.36it/s, loss=0.7777, iou=0.7570

In [ ]:
# 测试
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_results = test_model(model, val_loader, val_files, device)

In [ ]:
inference_example(model)